# 01 · Exploración de datos y EDA — EH2025

**Caso de estudio:** inasistencia escolar de personas de 6 a 17 años en Bolivia.  
**Fuente:** [INE — Encuesta de Hogares 2025](https://anda.ine.gob.bo/index.php/catalog/256).  
**Alcance:** análisis descriptivo, no identificación causal ni seguimiento longitudinal de deserción.

Este notebook **consulta** el dataset preparado por los scripts `src/audit_data.py`, `src/build_dataset.py`, `src/generate_eda.py`. No publica los microdatos ni modifica archivos de origen.


## 1. Configuración y trazabilidad

In [ ]:
from pathlib import Path

def project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "features.py").exists() and (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError("Abre VS Code en la raíz del proyecto EH2025 o ejecuta el notebook desde esa carpeta.")

ROOT = project_root()
FIG = ROOT / "outputs" / "figures"
MODELS = ROOT / "outputs" / "models"
print("Proyecto:", ROOT)

In [ ]:
import duckdb
import pandas as pd
import plotly.express as px

path = ROOT / "data" / "processed" / "dataset_escolar.parquet"
assert path.exists(), "Primero ejecuta python src/build_dataset.py"
conn = duckdb.connect(database=":memory:")
summary = conn.execute("""
    SELECT count(*) AS observaciones,
           count(DISTINCT folio) AS hogares,
           sum(inasistencia_escolar) AS casos_inasistencia,
           round(100.0 * avg(inasistencia_escolar), 3) AS tasa_muestral_no_ponderada,
           round(100.0 * sum(factor * inasistencia_escolar) / sum(factor), 3) AS tasa_ponderada
    FROM read_parquet(?)
""", [str(path)]).df()
display(summary)

## 2. Comprobar clases y calidad

La tasa **muestral no ponderada** y la **ponderada por factor de expansión** no deben confundirse. Las estimaciones ponderadas mostradas aquí no incluyen intervalos de confianza ajustados a estrato/UPM.

In [ ]:
counts = conn.execute("""
    SELECT inasistencia_escolar AS clase, count(*) AS registros
    FROM read_parquet(?)
    GROUP BY 1 ORDER BY 1
""", [str(path)]).df()
display(counts)
print("Suma de registros:", counts["registros"].sum())
assert counts["registros"].sum() == summary.loc[0, "observaciones"]

## 3. Comparaciones SQL ponderadas

In [ ]:
area = conn.execute("""
    SELECT CASE WHEN area=1 THEN 'Urbana' WHEN area=2 THEN 'Rural' ELSE 'Otra' END AS area,
           count(*) AS muestra,
           sum(inasistencia_escolar) AS casos,
           round(100 * sum(factor * inasistencia_escolar) / nullif(sum(factor),0), 3) AS tasa
    FROM read_parquet(?) GROUP BY 1 ORDER BY 1
""", [str(path)]).df()
display(area)
px.bar(area, x="area", y="tasa", text="tasa", title="Inasistencia escolar por área (ponderada)",
       labels={"tasa": "Tasa (%)", "area": "Área"}).show()

In [ ]:
edad = conn.execute("""
    SELECT cast(s01a_03 AS integer) AS edad,
           count(*) AS muestra,
           sum(inasistencia_escolar) AS casos,
           round(100 * sum(factor * inasistencia_escolar)/nullif(sum(factor),0),3) AS tasa
    FROM read_parquet(?) GROUP BY 1 ORDER BY 1
""", [str(path)]).df()
display(edad)
px.line(edad, x="edad", y="tasa", markers=True, title="Inasistencia por edad (ponderada)",
        labels={"tasa": "Tasa (%)", "edad": "Edad (años)"}).show()

## 4. Informe EDA completo

El informe automatizado de calidad se encuentra en `outputs/eda/reporte_eda_eh2025.html` y se genera con `python src/generate_eda.py`. Se recomienda revisar localmente cualquier contenido sensible antes de hacerlo público.

In [ ]:
from IPython.display import display, HTML
eda = ROOT / "outputs" / "eda" / "reporte_eda_eh2025.html"
print("EDA disponible:", eda.exists())
if eda.exists():
    display(HTML(f'<a href="{eda.as_uri()}" target="_blank">Abrir informe EDA local</a>'))
conn.close()

## 5. Interpretación

Documentar: (a) diferencias urbana/rural, (b) patrón por edad y (c) limitaciones del diseño muestral y de los pocos casos positivos. Las cifras no demuestran causalidad ni implican abandono definitivo.